# Fine-tuning Whisper-small for Hindi ASR

This notebook:
1. Downloads and preprocesses the training dataset (segment-level chunking)
2. Fine-tunes `openai/whisper-small` on Hindi audio
3. Evaluates baseline vs fine-tuned model on FLEURS Hindi test set using WER

## 0. Install Dependencies

In [ ]:
# Run once
!pip install -q transformers datasets librosa soundfile jiwer accelerate openpyxl

## 1. Load Metadata Sheet

In [ ]:
import pandas as pd

url = "https://docs.google.com/spreadsheets/d/1bujiO2NgtHlgqPlNvYAQf5_7ZcXARlIfNX5HNb9f8cI/export?format=xlsx"
df = pd.read_excel(url, engine="openpyxl")

print(f"Total recordings in sheet: {len(df)}")
df.head()

In [ ]:
# Rebuild GCS public URLs from the last two path segments
url_cols = ["rec_url_gcp", "transcription_url_gcp", "metadata_url_gcp"]

for col in url_cols:
    df[col] = (
        "https://storage.googleapis.com/upload_goai/"
        + df[col].str.extract(r'([^/]+/[^/]+)$')[0]
    )

print("Sample rec URL:", df["rec_url_gcp"].iloc[0])
print("Sample transcription URL:", df["transcription_url_gcp"].iloc[0])

## 2. Download Audio + Transcription Files

In [ ]:
import os
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

BASE_DIR = "data"
AUDIO_DIR = os.path.join(BASE_DIR, "audio")
TEXT_DIR  = os.path.join(BASE_DIR, "text")

os.makedirs(AUDIO_DIR, exist_ok=True)
os.makedirs(TEXT_DIR,  exist_ok=True)

MAX_WORKERS = 16


def download_file(url, path):
    if os.path.exists(path):
        return "skipped"
    try:
        r = requests.get(url, stream=True, timeout=30)
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        return "ok"
    except Exception as e:
        return f"error: {e}"


def process_row(row):
    rec_id     = str(row["recording_id"])
    audio_path = os.path.join(AUDIO_DIR, f"{rec_id}.wav")
    text_path  = os.path.join(TEXT_DIR,  f"{rec_id}.json")
    a_status   = download_file(row["rec_url_gcp"],           audio_path)
    t_status   = download_file(row["transcription_url_gcp"], text_path)
    return rec_id, a_status, t_status


results = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(process_row, row) for _, row in df.iterrows()]
    for future in as_completed(futures):
        result = future.result()
        results.append(result)

ok      = sum(1 for _, a, t in results if a == "ok" and t == "ok")
skipped = sum(1 for _, a, t in results if a == "skipped" or t == "skipped")
errors  = sum(1 for _, a, t in results if "error" in a or "error" in t)

print(f"Downloaded: {ok} | Skipped (cached): {skipped} | Errors: {errors}")

## 3. Preprocessing: Segment-Level Chunking

**Key design choice:** Each transcription JSON contains a list of timestamped segments:
```json
[{"start": 0.0, "end": 4.5, "text": "..."},  ...]
```
Rather than concatenating all segment texts (which causes the `labels > 448 tokens` error),
we slice the audio array using each segment's start/end timestamps and pair it with
that segment's text. This produces short (0.5–30 s) audio–text pairs suitable for Whisper.

**Steps applied:**
1. Load full WAV at 16 kHz with librosa
2. Parse JSON → iterate segments
3. Slice `audio[start_sample : end_sample]`
4. Filter: skip if duration < 0.5 s or > 30 s, or if text is empty
5. Collect `(numpy_array, text)` pairs

In [ ]:
import json
import librosa
import numpy as np

audio_chunks   = []
transcriptions = []

MIN_DURATION = 0.5   # seconds
MAX_DURATION = 30.0  # Whisper's native context window

for file in os.listdir(AUDIO_DIR):
    rec_id     = os.path.splitext(file)[0]
    audio_path = os.path.join(AUDIO_DIR, file)
    text_path  = os.path.join(TEXT_DIR,  f"{rec_id}.json")

    if not os.path.exists(text_path):
        continue

    try:
        # Load entire recording at 16 kHz
        audio_array, sr = librosa.load(audio_path, sr=16000)

        with open(text_path, "r", encoding="utf-8") as f:
            segments = json.load(f)

        if not isinstance(segments, list):
            continue

        for seg in segments:
            start = seg.get("start")
            end   = seg.get("end")
            text  = seg.get("text", "").strip()

            # Skip malformed or empty segments
            if not text or start is None or end is None:
                continue

            duration = end - start
            if duration < MIN_DURATION or duration > MAX_DURATION:
                continue

            # Slice audio using sample indices
            start_sample = int(start * sr)
            end_sample   = int(end   * sr)
            chunk        = audio_array[start_sample:end_sample]

            # Safety check: skip if chunk is empty
            if len(chunk) == 0:
                continue

            audio_chunks.append(chunk)
            transcriptions.append(text)

    except Exception as e:
        print(f"Error processing {rec_id}: {e}")
        continue

print(f"Total (audio, text) pairs: {len(audio_chunks)}")
if audio_chunks:
    durs = [(len(c) / 16000) for c in audio_chunks]
    print(f"Chunk duration — min: {min(durs):.1f}s | max: {max(durs):.1f}s | mean: {np.mean(durs):.1f}s")
    print(f"Sample text: {transcriptions[0][:120]}")

## 4. Build HuggingFace Dataset & Train/Val/Test Split

In [ ]:
from datasets import Dataset

data = Dataset.from_dict({
    "audio": audio_chunks,   # numpy arrays (already loaded at 16 kHz)
    "text":  transcriptions
})

# 80 / 10 / 10 split
split1     = data.train_test_split(test_size=0.2, seed=42)
split2     = split1["test"].train_test_split(test_size=0.5, seed=42)

train_data = split1["train"]
val_data   = split2["train"]
test_data  = split2["test"]

print(f"Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")

## 5. Load Processor & Feature Extraction

In [ ]:
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained(
    "openai/whisper-small",
    language="hi",
    task="transcribe"
)


def prepare(batch):
    """Convert raw numpy audio + text into model-ready features + labels."""
    audio_array = batch["audio"]   # already a numpy array at 16 kHz

    # Audio → log-mel spectrogram (80 mel bins, 3000 frames)
    inputs = processor.feature_extractor(
        audio_array,
        sampling_rate=16000,
        return_tensors="np"
    )

    # Text → token ids; truncate to Whisper's 448-token decoder limit
    labels = processor.tokenizer(
        batch["text"],
        truncation=True,
        max_length=448
    ).input_ids

    return {
        "input_features": inputs["input_features"][0],
        "labels":         labels
    }

In [ ]:
train_data = train_data.map(prepare, remove_columns=train_data.column_names, num_proc=4)
val_data   = val_data.map(prepare,   remove_columns=val_data.column_names,   num_proc=4)

print("Preprocessed train:", train_data)
print("Preprocessed val:",   val_data)

## 6. Data Collator

In [ ]:
import torch


def data_collator(features):
    input_features = [{"input_features": f["input_features"]} for f in features]
    labels         = [f["labels"] for f in features]

    # Pad log-mel features to the same length
    batch = processor.feature_extractor.pad(input_features, return_tensors="pt")

    # Pad token labels; replace pad token id with -100 so loss ignores them
    labels_batch = processor.tokenizer.pad(
        {"input_ids": labels}, return_tensors="pt"
    )
    labels_padded = labels_batch["input_ids"].masked_fill(
        labels_batch["attention_mask"] != 1, -100
    )

    batch["labels"] = labels_padded
    return batch

## 7. Load Model & Configure for Hindi

In [ ]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

# Use generation_config (replaces deprecated forced_decoder_ids on model.config)
model.generation_config.language = "hindi"
model.generation_config.task     = "transcribe"
model.generation_config.forced_decoder_ids = None

# Suppress blank/silence tokens to reduce empty predictions
model.config.suppress_tokens = []

print("Model loaded. Parameters:", sum(p.numel() for p in model.parameters()) // 1_000_000, "M")

## 8. WER Metric

In [ ]:
import numpy as np
from jiwer import wer as compute_wer


def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids

    # Replace -100 back to pad token id before decoding
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str  = processor.tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer_score = compute_wer(label_str, pred_str)
    return {"wer": round(wer_score, 4)}

## 9. Training Arguments & Trainer

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir                  = "./whisper-hi",
    per_device_train_batch_size = 8,
    gradient_accumulation_steps = 2,          # effective batch = 16
    learning_rate               = 1e-5,
    warmup_steps                = 500,
    max_steps                   = 5000,
    eval_strategy               = "steps",
    eval_steps                  = 1000,       # ← was missing in original
    save_steps                  = 1000,
    logging_steps               = 100,
    predict_with_generate       = True,
    generation_max_length       = 448,
    fp16                        = True,
    load_best_model_at_end      = True,
    metric_for_best_model       = "wer",
    greater_is_better           = False,
    report_to                   = "none",
)

In [ ]:
trainer = Seq2SeqTrainer(
    model            = model,
    args             = training_args,
    train_dataset    = train_data,
    eval_dataset     = val_data,
    processing_class = processor,
    data_collator    = data_collator,
    compute_metrics  = compute_metrics,   # ← was missing in original
)

In [ ]:
trainer.train()

In [ ]:
trainer.save_model("./whisper-hi-final")
processor.save_pretrained("./whisper-hi-final")
print("Model saved to ./whisper-hi-final")

## 10. Evaluation on FLEURS Hindi Test Set

We load the official FLEURS Hindi test split and evaluate:
- **Baseline**: pretrained `openai/whisper-small` (no fine-tuning)
- **Fine-tuned**: our `./whisper-hi-final` checkpoint

Metric: **Word Error Rate (WER)** — lower is better.

In [ ]:
from datasets import load_dataset

fleurs_test = load_dataset(
    "google/fleurs",
    "hi_in",
    split="test",
    trust_remote_code=True
)

print(f"FLEURS Hindi test samples: {len(fleurs_test)}")
print(fleurs_test[0].keys())

In [ ]:
import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from jiwer import wer as compute_wer
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"


def evaluate_model(model_name_or_path, fleurs_dataset, desc=""):
    """Run inference on FLEURS Hindi test set and return WER."""
    print(f"\n{'='*50}")
    print(f"Evaluating: {desc or model_name_or_path}")
    print(f"{'='*50}")

    proc  = WhisperProcessor.from_pretrained(model_name_or_path, language="hi", task="transcribe")
    mdl   = WhisperForConditionalGeneration.from_pretrained(model_name_or_path).to(device)
    mdl.eval()

    forced_ids = proc.get_decoder_prompt_ids(language="hi", task="transcribe")

    all_preds  = []
    all_labels = []

    BATCH_SIZE = 16

    for i in range(0, len(fleurs_dataset), BATCH_SIZE):
        batch = fleurs_dataset[i : i + BATCH_SIZE]

        # FLEURS audio is already at 16 kHz
        audio_arrays = [np.array(a["array"]) for a in batch["audio"]]
        references   = batch["transcription"]

        inputs = proc.feature_extractor(
            audio_arrays,
            sampling_rate=16000,
            return_tensors="pt",
            padding=True
        ).to(device)

        with torch.no_grad():
            generated = mdl.generate(
                **inputs,
                forced_decoder_ids=forced_ids,
                max_new_tokens=448
            )

        preds = proc.tokenizer.batch_decode(generated, skip_special_tokens=True)

        all_preds.extend(preds)
        all_labels.extend(references)

        if (i // BATCH_SIZE) % 5 == 0:
            print(f"  Processed {min(i+BATCH_SIZE, len(fleurs_dataset))}/{len(fleurs_dataset)} samples...")

    wer_score = compute_wer(all_labels, all_preds)
    print(f"\n  ✅ WER: {wer_score*100:.2f}%")

    # Show a few examples
    print("\n  Sample predictions:")
    for ref, pred in zip(all_labels[:3], all_preds[:3]):
        print(f"    REF : {ref}")
        print(f"    PRED: {pred}")
        print()

    return wer_score, all_preds, all_labels

In [ ]:
# --- Baseline: pretrained whisper-small (no fine-tuning) ---
baseline_wer, baseline_preds, refs = evaluate_model(
    "openai/whisper-small",
    fleurs_test,
    desc="Baseline: openai/whisper-small (pretrained)"
)

In [ ]:
# --- Fine-tuned checkpoint ---
finetuned_wer, finetuned_preds, _ = evaluate_model(
    "./whisper-hi-final",
    fleurs_test,
    desc="Fine-tuned: whisper-small on Hindi custom dataset"
)

In [ ]:
# --- Summary Table ---
print("\n" + "="*50)
print("EVALUATION SUMMARY — FLEURS Hindi Test Set")
print("="*50)
print(f"{'Model':<40} {'WER':>8}")
print("-"*50)
print(f"{'Baseline (whisper-small)':<40} {baseline_wer*100:>7.2f}%")
print(f"{'Fine-tuned (whisper-small + Hindi data)':<40} {finetuned_wer*100:>7.2f}%")
print("-"*50)

improvement = baseline_wer - finetuned_wer
if improvement > 0:
    print(f"Relative improvement: {improvement*100:.2f} pp (↓ WER)")
else:
    print(f"No improvement: {abs(improvement)*100:.2f} pp (↑ WER)")